# Week 3: Baseline Predictive Modeling (Churn Prediction)

In this notebook, we will:
1. Build a Customer Segmentation Model (Basic grouping by booking behavior).
2. Prepare data for machine learning (Encoding categorical variables).
3. Train a baseline Logistic Regression model to predict `is_canceled`.
4. Evaluate the model using Accuracy, Precision, Recall, and ROC-AUC.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, roc_curve, classification_report

%matplotlib inline

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/cleaned_hotel_bookings.csv')

## 2. Customer Segmentation (Simple)
Grouping users based on Corporate vs Leisure, and early planners vs last-minute.

In [ ]:
df['planner_type'] = np.where(df['lead_time'] > 30, 'Early Planner', 'Last Minute')

plt.figure(figsize=(8,5))
sns.countplot(x='market_segment', hue='planner_type', data=df)
plt.title('Customer Segmentation: Market Segment vs Planner Type')
plt.xticks(rotation=45)
plt.show()

## 3. Data Preparation for Machine Learning

In [ ]:
# Select subset of features for baseline model to keep it simple and interpretable
features = ['lead_time', 'total_duration', 'previous_cancellations', 
            'booking_changes', 'adr', 'total_of_special_requests', 'market_segment', 'deposit_type']

X = df[features].copy()
y = df['is_canceled']

# One-hot encode categorical features
X = pd.get_dummies(X, columns=['market_segment', 'deposit_type'], drop_first=True)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scale numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Train Logistic Regression Model

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

## 5. Model Evaluation

In [ ]:
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

print('Classification Report:')
print(classification_report(y_test, y_pred))

print(f'Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}')

## 6. ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {roc_auc_score(y_test, y_proba):.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.show()